In [ ]:
import pandas as pd
import numpy as np
import os

def create_fact_sales():
    print("--- Đang tải các bảng chiều (Dimension Tables) ---")
    path_date     = '/content/dim_date_new.csv'          if os.path.exists('/content/dim_date_new.csv')          else '/content/dim_date.csv'
    path_customer = '/content/dim_customers_new.csv'     if os.path.exists('/content/dim_customers_new.csv')     else '/content/dim_customers.csv'
    path_product  = '/content/dim_product_new.csv'       if os.path.exists('/content/dim_product_new.csv')       else '/content/dim_product.csv'
    path_geo      = '/content/dim_geography_new.csv'     if os.path.exists('/content/dim_geography_new.csv')     else '/content/dim_geography.csv'
    path_employee = '/content/dim_sales_employee_new.csv' if os.path.exists('/content/dim_sales_employee_new.csv') else '/content/dim_sales_employee.csv'
    path_shipper  = '/content/dim_shipper_new.csv'       if os.path.exists('/content/dim_shipper_new.csv')       else '/content/dim_shipper.csv'
    path_promotion= '/content/dim_promotion_new.csv'     if os.path.exists('/content/dim_promotion_new.csv')     else '/content/dim_promotion.csv'

    dim_date      = pd.read_csv(path_date)
    dim_customer  = pd.read_csv(path_customer)
    dim_product   = pd.read_csv(path_product)
    dim_geo       = pd.read_csv(path_geo)
    dim_employee  = pd.read_csv(path_employee)
    dim_shipper   = pd.read_csv(path_shipper)
    dim_promotion = pd.read_csv(path_promotion)

    # ── FIX #1: dim_customers là SCD Type 2 → mỗi customer_id có nhiều dòng (nhiều phiên bản lịch sử).
    #    Chỉ giữ lại bản ghi MỚI NHẤT (current record) để tránh fanout many-to-many khi join.
    dim_customer_current = (
        dim_customer
        .sort_values('customer_key', ascending=False)   # customer_key lớn hơn = bản ghi mới hơn
        .drop_duplicates(subset=['customer_id'], keep='first')
    )
    print(f"[Info] dim_customer: {len(dim_customer)} dòng → sau dedup còn {len(dim_customer_current)} dòng (1 per customer_id)")

    print("--- Đang tải dữ liệu giao dịch nguồn ---")
    orders      = pd.read_csv('/content/orders_new.csv')
    shipments   = pd.read_csv('/content/shipments_new.csv')
    order_items = pd.read_csv('/content/order_item_new.csv')
    payments    = pd.read_csv('/content/payments_new.csv')

    # 1. Kết hợp dữ liệu giao dịch
    fact_df = pd.merge(order_items, orders,    on='order_id',                       how='inner')
    fact_df = pd.merge(fact_df,    shipments,  on='order_id',                       how='left')
    fact_df = pd.merge(fact_df,    payments,   left_on='order_id', right_on='order_id_FK', how='left')
    print(f"[Info] Sau khi join fact chính: {len(fact_df)} dòng")

    # 2. Ánh xạ các trường Date sang Date Key (YYYYMMDD)
    for date_col in ['order_date', 'ship_date', 'delivery_date']:
        if date_col in fact_df.columns:
            fact_df[date_col + '_dt'] = pd.to_datetime(fact_df[date_col], errors='coerce')
        else:
            fact_df[date_col + '_dt'] = pd.NaT

    fact_df['order_date_key']    = fact_df['order_date_dt'].dt.strftime('%Y%m%d').astype(float).fillna(-1).astype(int)
    fact_df['ship_date_key']     = fact_df['ship_date_dt'].dt.strftime('%Y%m%d').astype(float).fillna(-1).astype(int)
    fact_df['delivery_date_key'] = fact_df['delivery_date_dt'].dt.strftime('%Y%m%d').astype(float).fillna(-1).astype(int)

    # 3. Ánh xạ Customer Key từ DIM_CUSTOMER (dùng bản đã dedup — FIX #1)
    fact_df = pd.merge(fact_df, dim_customer_current[['customer_id', 'customer_key']],
                       left_on='customer_id_FK', right_on='customer_id', how='left')
    fact_df['customer_key'] = fact_df['customer_key'].fillna(-1).astype(int)
    print(f"[Info] Sau merge dim_customer: {len(fact_df)} dòng")

    # 4. Ánh xạ Product Key VÀ lấy current_cogs trong MỘT lần join duy nhất (FIX #2)
    #    (Tránh merge dim_product lần 2 ở bước COGS — đó là nguyên nhân tiềm ẩn fanout)
    product_cols = ['product_id', 'product_key']
    if 'current_cogs' in dim_product.columns:
        product_cols.append('current_cogs')
    fact_df = pd.merge(fact_df, dim_product[product_cols], on='product_id', how='left')
    fact_df['product_key'] = fact_df['product_key'].fillna(-1).astype(int)
    if 'current_cogs' in fact_df.columns:
        fact_df['cogs'] = (fact_df['current_cogs'] * fact_df['quantity']).fillna(0.0)
    else:
        fact_df['cogs'] = 0.0

    # 5. Ánh xạ Geography Key từ DIM_GEOGRAPHY
    #    Dùng dim_customer_current (đã dedup) để tránh fanout lần 2 khi tạo customer_geo
    customer_geo = pd.merge(dim_customer_current, dim_geo, on='zip', how='left', suffixes=('', '_geo'))
    fact_df = pd.merge(fact_df, customer_geo[['customer_key', 'geography_key']], on='customer_key', how='left')
    fact_df = fact_df.rename(columns={'geography_key': 'order_geo_key'})
    fact_df['order_geo_key'] = fact_df['order_geo_key'].fillna(-1).astype(int)

    # 6. Ánh xạ Sales Employee Key
    emp_col = ('sales_employee_id_FK' if 'sales_employee_id_FK' in fact_df.columns
               else ('sales_employee_id' if 'sales_employee_id' in fact_df.columns else None))
    if emp_col:
        fact_df = pd.merge(fact_df, dim_employee[['sales_employee_id', 'sales_employee_key']],
                           left_on=emp_col, right_on='sales_employee_id', how='left')
    else:
        fact_df['sales_employee_key'] = -1
    fact_df['sales_employee_key'] = fact_df['sales_employee_key'].fillna(-1).astype(int)

    # 7. Ánh xạ Shipper Key
    ship_col = 'shipper_id' if 'shipper_id' in fact_df.columns else None
    if ship_col:
        fact_df = pd.merge(fact_df, dim_shipper[['shipper_id', 'shipper_key']], on='shipper_id', how='left')
    else:
        fact_df['shipper_key'] = -1
    fact_df['shipper_key'] = fact_df['shipper_key'].fillna(-1).astype(int)

    # 8. Ánh xạ Promo Keys
    promo_col1 = 'promo_id'   if 'promo_id'   in fact_df.columns else None
    promo_col2 = 'promo_id_2' if 'promo_id_2' in fact_df.columns else None

    if promo_col1:
        fact_df = pd.merge(fact_df, dim_promotion[['promo_id', 'promo_key']],
                           left_on=promo_col1, right_on='promo_id', how='left', suffixes=('', '_p1'))
        fact_df = fact_df.rename(columns={'promo_key': 'promo_key_1'})
    else:
        fact_df['promo_key_1'] = -1
    fact_df['promo_key_1'] = fact_df['promo_key_1'].fillna(-1).astype(int)

    if promo_col2:
        fact_df = pd.merge(fact_df, dim_promotion[['promo_id', 'promo_key']],
                           left_on=promo_col2, right_on='promo_id', how='left', suffixes=('', '_p2'))
        fact_df = fact_df.rename(columns={'promo_key': 'promo_key_2'})
    else:
        fact_df['promo_key_2'] = -1
    fact_df['promo_key_2'] = fact_df['promo_key_2'].fillna(-1).astype(int)

    # 9. Tính toán Measures & Degenerate Dimensions
    fact_df['sales_fact_id'] = range(1, len(fact_df) + 1)

    delivery_dur = (fact_df['delivery_date_dt'] - fact_df['order_date_dt']).dt.total_seconds() / (24 * 3600)
    fact_df['delivery_duration_days'] = delivery_dur.round(2).fillna(0)

    # Tính gross/net nếu chưa có sẵn
    measures_cols = [
        'quantity', 'unit_price', 'discount_amount',
        'gross_sales_amount', 'net_sales_amount', 'shipping_fee',
        'payment_value', 'installments'
    ]
    for col in measures_cols:
        if col not in fact_df.columns:
            if col == 'gross_sales_amount':
                fact_df['gross_sales_amount'] = fact_df['quantity'] * fact_df['unit_price']
            elif col == 'net_sales_amount':
                fact_df['net_sales_amount'] = fact_df['gross_sales_amount'] - fact_df.get('discount_amount', 0)
            else:
                fact_df[col] = 0.0

    deg_cols = ['order_id', 'order_status', 'device_type', 'order_source', 'payment_method']
    for col in deg_cols:
        if col not in fact_df.columns:
            fact_df[col] = 'Unknown'

    # 10. Lựa chọn và xuất đúng schema
    fact_sales_columns = [
        'sales_fact_id',
        'order_date_key', 'ship_date_key', 'delivery_date_key',
        'customer_key', 'product_key', 'order_geo_key',
        'sales_employee_key', 'shipper_key', 'promo_key_1', 'promo_key_2',
        'order_id', 'order_status', 'device_type', 'order_source', 'payment_method',
        'quantity', 'unit_price', 'cogs', 'discount_amount',
        'gross_sales_amount', 'net_sales_amount', 'shipping_fee',
        'payment_value', 'installments', 'delivery_duration_days'
    ]

    for col in fact_sales_columns:
        if col not in fact_df.columns:
            fact_df[col] = np.nan

    fact_sales = fact_df[fact_sales_columns].copy()

    # Làm tròn các cột số thập phân về 2 chữ số để giảm kích thước file CSV
    decimal_cols = [
        'cogs', 'unit_price', 'discount_amount',
        'gross_sales_amount', 'net_sales_amount',
        'shipping_fee', 'payment_value', 'delivery_duration_days'
    ]
    for col in decimal_cols:
        if col in fact_sales.columns:
            fact_sales[col] = fact_sales[col].round(2)

    output_path = '/content/fact_sales.csv'
    fact_sales.to_csv(output_path, index=False)
    print(f"[Thành công] Đã xây dựng bảng FACT_SALES với {len(fact_sales)} dòng dữ liệu.")
    print(f"File kết quả đã được lưu tại: {output_path}")
    return fact_sales

fact_sales_df = create_fact_sales()
display(fact_sales_df.head())
